# Temporal XOR Evidence Accumulation

Tests whether the cross-depth eligibility trace (**ε_z**) in Millidge (2025) deep e-prop carries useful gradient signal when the task structure **requires** two serial processing stages.

**Task**: N=7 cues, each spanning w=2 consecutive steps.
- Step 0 of cue: x₀ ∈ {-1, +1} on channel 0
- Step 1 of cue: x₁ ∈ {-1, +1} on channel 1
- Cue label = XOR(sign x₀, sign x₁) — same sign → “right”, opposite → “left”
- Trial label = majority vote of N cue labels — decision after silent delay D.

**Why depth matters**: A depth-1 RNN must use its single hidden layer for (a) buffering x₀ one step, (b) computing XOR at step 1, and (c) integrating evidence across the delay — three competing uses of the same capacity. A depth-2 RNN separates timescales: bottom layer buffers x₀, top layer runs the long-delay accumulator. The cross-depth trace (ε_z) carries gradient credit from the accumulator back to the buffer.

**Experiments**
1. Gate 1 — depth screen: BPTT depth=2 > depth=1 by ≥ threshold
2. Training curves: BPTT, deep e-prop, d=0, no_ε_z at fixed delay
3. **PRIMARY** — layer-resolved cosine vs BPTT as function of delay D (deep e-prop vs no_ε_z)


In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    # TODO: replace with your repo URL
    subprocess.run(['git', 'clone', 'https://github.com/YOUR_USER/NeuroAI.git'], check=True)
    for root, dirs, _ in os.walk('.'):
        if 'models' in dirs and 'learning_rules' in dirs and 'tasks' in dirs:
            sys.path.insert(0, root)
            print(f'Project root: {root}')
            break
else:
    sys.path.insert(0, os.getcwd())

# Clear stale module cache after git pull
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ('models.', 'learning_rules.', 'tasks.')):
        del sys.modules[mod]

print('Setup complete.')


In [ ]:
import json, math, time
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor

from models.deep_rnn import DeepRNN
from learning_rules.bptt import compute_bptt_gradients, _trace_mse_loss
from learning_rules.deep_eprop import compute_deep_eprop_gradients, mse_error
import tasks.temporal_xor_accumulation as TXA

plt.style.use('seaborn-v0_8-whitegrid')
DEVICE = 'cpu'   # temporal XOR is fast on CPU; switch to 'cuda' if n_rec is large
print('Device:', DEVICE)


## Configuration

Set `RUN_PRESET = 'colab'` for the full experiment. `'smoke'` completes in under a minute to verify the notebook runs end-to-end.


In [ ]:
RUN_PRESET = 'smoke'   # <- change to 'colab' for the full run

PRESETS = {
    'smoke': dict(
        n_cues=7, w=2, gap=2, noise_level=0.05,
        fixed_delay=5,
        delays_cosine=[0, 5, 10],
        n_rec=32,
        batch_size=32, eval_batch_size=128,
        n_steps=100, eval_every=25, n_seeds=2, lr=1e-2,
        n_steps_gate=100, n_seeds_gate=2, d2_acc_threshold=0.65,
        n_trials_cosine=3,
    ),
    'colab': dict(
        n_cues=7, w=2, gap=2, noise_level=0.05,
        fixed_delay=20,
        delays_cosine=[0, 5, 10, 20, 40],
        n_rec=64,
        batch_size=128, eval_batch_size=512,
        n_steps=3000, eval_every=100, n_seeds=5, lr=5e-3,
        n_steps_gate=2000, n_seeds_gate=5, d2_acc_threshold=0.65,
        n_trials_cosine=10,
    ),
}

CFG = PRESETS[RUN_PRESET]

TASK_KW = dict(
    n_cues      = CFG['n_cues'],
    w           = CFG['w'],
    gap         = CFG['gap'],
    noise_level = CFG['noise_level'],
    delay       = CFG['fixed_delay'],
)
N_IN  = TXA.N_IN
N_OUT = TXA.N_OUT

RESULTS_DIR = Path('results') / f'temporal_xor_{RUN_PRESET}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Preset : {RUN_PRESET}   device : {DEVICE}')
print(f'Task   : {TASK_KW}')
print(f'Results: {RESULTS_DIR}')


## Task sanity check

Verify shapes, mask placement, label balance (≈50 / 50), and visualise one trial.


In [ ]:
ex_inp, ex_tgt, ex_msk = TXA.generate_batch(batch_size=8, seed=0, **TASK_KW)
T     = ex_inp.shape[0]
t_rec = TXA.decision_timestep(
    n_cues=TASK_KW['n_cues'], delay=TASK_KW['delay'],
    w=TASK_KW['w'], gap=TASK_KW['gap'])

assert ex_inp.shape  == (T, 8, N_IN),  f'input shape {ex_inp.shape}'
assert ex_tgt.shape  == (T, 8, N_OUT), f'target shape {ex_tgt.shape}'
assert ex_msk.shape  == (T, 8),        f'mask shape {ex_msk.shape}'
assert float(ex_msk[t_rec].mean())  == 1.0, 'mask must be 1 at decision step'
assert float(ex_msk[:t_rec].sum()) == 0.0,  'mask must be 0 before decision step'

_, big_tgt, _ = TXA.generate_batch(batch_size=4000, seed=1, **TASK_KW)
p_right = big_tgt[t_rec].argmax(-1).float().mean().item()
assert abs(p_right - 0.5) < 0.04, f'Label imbalance: P(right)={p_right:.3f}'

print(f'T={T}  t_recall={t_rec}  N_IN={N_IN}')
print(f'P(right) over 4000 trials: {p_right:.3f}  (expect 0.50 \u00b1 0.04)')
print('Task sanity PASSED')

# Visualise one trial
fig, axes = plt.subplots(3, 1, figsize=(11, 5), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1.2, 0.6]})
axes[0].imshow(ex_inp[:, 0, :].T.numpy(), aspect='auto',
               interpolation='nearest', cmap='RdBu_r', vmin=-1.2, vmax=1.2)
axes[0].set_ylabel('Input channel')
axes[0].set_yticks(range(N_IN))
axes[0].set_yticklabels(['feat x0', 'feat x1', 'recall', 'noise', 'bias'], fontsize=8)
axes[0].axvline(t_rec, color='k', linestyle='--', linewidth=1.2, label='decision')
axes[0].legend(fontsize=8, loc='upper right')
axes[1].imshow(ex_tgt[:, 0, :].T.numpy(), aspect='auto',
               interpolation='nearest', cmap='Greens')
axes[1].set_ylabel('Target')
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(['left', 'right'], fontsize=8)
axes[2].plot(ex_msk[:, 0].numpy(), 'k')
axes[2].set_ylabel('Mask'); axes[2].set_xlabel('Timestep')
axes[2].set_ylim(-0.05, 1.2)
fig.suptitle('Trial 0 \u2014 temporal XOR evidence accumulation', fontsize=11)
fig.tight_layout()
fig.savefig(RESULTS_DIR / f'task_example_{RUN_PRESET}.svg')
plt.show()


## Gate 1 — Depth screen

Train BPTT at depth=1 and depth=2. Accept iff depth=2 final accuracy exceeds depth=1 by ≥ `depth_gap_threshold` (mean over seeds).

If Gate 1 fails, increase `n_rec` or `n_steps_gate` in the preset before running the main experiment.


In [ ]:
def make_model(n_layers, n_rec, n_in, n_out, seed):
    torch.manual_seed(seed)
    return DeepRNN(n_in, n_rec, n_out, n_layers=n_layers).to(DEVICE)


def apply_grads(model, grads, lr, clip_norm=10.0):
    all_g = torch.cat([g.detach().flatten() for g in grads.values()])
    norm  = all_g.norm().item()
    scale = min(1.0, clip_norm / (norm + 1e-12))
    for name, param in model.named_parameters():
        if name in grads:
            param.data -= lr * scale * grads[name].to(param.device)


def run_depth_screen(task_kw, n_seeds, n_steps, n_rec, batch_size,
                     eval_batch_size, lr, eval_every):
    results = {'d1_runs': [], 'd2_runs': []}
    for depth, key in [(1, 'd1_runs'), (2, 'd2_runs')]:
        for seed in range(n_seeds):
            model = make_model(depth, n_rec, N_IN, N_OUT, seed)
            steps, accs = [], []
            for step in range(1, n_steps + 1):
                inp, tgt, msk = TXA.generate_batch(
                    batch_size, seed=seed * 1_000_003 + step, device=DEVICE, **task_kw)
                grads = compute_bptt_gradients(model, inp, tgt, msk, _trace_mse_loss)
                apply_grads(model, grads, lr)
                if step % eval_every == 0 or step == n_steps:
                    with torch.no_grad():
                        ei, et, em = TXA.generate_batch(
                            eval_batch_size, seed=9_000_000 + step, device=DEVICE, **task_kw)
                        out, _ = model(ei)
                        acc = TXA.task_accuracy(out, et, em)
                    steps.append(step)
                    accs.append(acc)
            results[key].append({'seed': seed, 'steps': steps, 'accs': accs,
                                  'final_acc': accs[-1]})
            print(f'  depth={depth}  seed={seed:2d}  final acc={accs[-1]:.3f}')

    d1_mean = float(np.mean([r['final_acc'] for r in results['d1_runs']]))
    d2_mean = float(np.mean([r['final_acc'] for r in results['d2_runs']]))
    d2_frac = float(np.mean([r['final_acc'] > CFG['d2_acc_threshold']
                             for r in results['d2_runs']]))
    passed  = d2_frac >= 0.5   # majority of seeds beat absolute threshold
    results.update({'d1_mean': d1_mean, 'd2_mean': d2_mean,
                    'd2_frac': d2_frac, 'passed': passed})

    thresh = CFG['d2_acc_threshold']
    print(f'\nGate 1:  depth=1={d1_mean:.3f}   depth=2={d2_mean:.3f}')
    print(f'         depth=2 seeds above {thresh:.2f}: {d2_frac:.0%}  (need >= 50%)')
    print('\u2192', 'PASSED' if passed else 'FAILED \u2014 increase n_rec or n_steps_gate')
    return results


print('Running Gate 1 \u2014 depth screen ...')
t0 = time.time()
screen_result = run_depth_screen(
    task_kw=TASK_KW, n_seeds=CFG['n_seeds_gate'],
    n_steps=CFG['n_steps_gate'], n_rec=CFG['n_rec'],
    batch_size=CFG['batch_size'], eval_batch_size=CFG['eval_batch_size'],
    lr=CFG['lr'], eval_every=CFG['eval_every'],
)
print(f'Gate 1 done in {time.time() - t0:.1f}s')
with open(RESULTS_DIR / f'gate1_{RUN_PRESET}.json', 'w') as f:
    json.dump(screen_result, f, indent=2)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for depth, key, color in [(1, 'd1_runs', '#4C78A8'), (2, 'd2_runs', '#54A24B')]:
    runs = screen_result[key]
    mat  = np.array([r['accs'] for r in runs])
    xs   = runs[0]['steps']
    mn   = mat.mean(0)
    se   = mat.std(0, ddof=1) / np.sqrt(len(mat)) if len(mat) > 1 else np.zeros_like(mn)
    ax.plot(xs, mn, color=color, linewidth=2, label=f'depth={depth}')
    ax.fill_between(xs, mn - se, mn + se, color=color, alpha=0.2)

ax.axhline(0.5, color='0.5', linestyle=':', linewidth=1, label='chance')
ax.set_xlabel('Training step'); ax.set_ylabel('Decision accuracy')
ax.set_title(f'Gate 1 depth screen \u2014 gap={screen_result["gap"]:.3f} '
             f'({"PASSED" if screen_result["passed"] else "FAILED"})')
ax.legend(); ax.set_ylim(0.3, 1.05)
fig.tight_layout()
fig.savefig(RESULTS_DIR / f'gate1_{RUN_PRESET}.svg')
plt.show()


## Helper functions

- **`compute_no_eps_z_gradients`**: deep e-prop with the cross-depth trace (ε_z) zeroed. All lower-layer parameter gradients are set to zero, which is equivalent to removing `eps_cross` from `deep_eprop.py`. Distinct from `d=0` (which drops temporal carry but keeps the spatial eps_cross term).
- **`layer_groups`** / **`cosine_for_keys`**: group parameter keys by layer and compute cosine similarity between gradient dicts.


In [ ]:
def compute_no_eps_z_gradients(model, inputs, targets, mask, learning_signal_fn=mse_error):
    grads = compute_deep_eprop_gradients(
        model, inputs, targets, mask, learning_signal_fn, d_zero=False)
    L = model.n_layers
    for l in range(L - 1):
        grads[f'W_recs.{l}'] = torch.zeros_like(grads[f'W_recs.{l}'])
        grads[f'biases.{l}'] = torch.zeros_like(grads[f'biases.{l}'])
        if l == 0:
            grads['W_in'] = torch.zeros_like(grads['W_in'])
        else:
            grads[f'W_ffs.{l - 1}'] = torch.zeros_like(grads[f'W_ffs.{l - 1}'])
    return grads


def layer_groups(n_layers: int) -> Dict[str, List[str]]:
    groups: Dict[str, List[str]] = {}
    hidden_all: List[str] = []
    for l in range(n_layers):
        keys = [f'W_recs.{l}', f'biases.{l}']
        keys.append('W_in' if l == 0 else f'W_ffs.{l - 1}')
        groups[f'layer{l + 1}'] = keys
        hidden_all.extend(keys)
    groups['hidden_all'] = hidden_all
    groups['readout']    = ['W_out', 'b_out']
    groups['all']        = hidden_all + groups['readout']
    if n_layers >= 1:
        groups['bottom'] = groups['layer1']
        groups['top']    = groups[f'layer{n_layers}']
    return groups


GROUPS_2L = layer_groups(2)


def cosine_for_keys(ga, gb, keys, eps=1e-12):
    shared = [k for k in keys if k in ga and k in gb]
    if not shared:
        return float('nan')
    va = torch.cat([ga[k].detach().flatten() for k in shared])
    vb = torch.cat([gb[k].detach().flatten() for k in shared])
    na, nb = va.norm().item(), vb.norm().item()
    if na < eps or nb < eps:
        return float('nan')
    return float((va @ vb / (na * nb)).item())


print('Helpers defined.')
print('Layer groups:', {k: GROUPS_2L[k] for k in ('bottom', 'top', 'hidden_all')})


## Training curves — all four rules

Trains BPTT, deep e-prop, d=0, and no_ε_z at the fixed delay. `no_eps_z` zeroes all bottom-layer gradients; it should fail to train the bottom layer and underperform deep e-prop, demonstrating that ε_z is load-bearing.


In [ ]:
RULE_CFG = {
    'bptt':       ('BPTT',          '#4C78A8', None),
    'deep-eprop': ('Deep e-prop',   '#54A24B',
                   lambda m, i, t, k: compute_deep_eprop_gradients(
                       m, i, t, k, mse_error, d_zero=False)),
    'd=0':        ('d=0',           '#E45756',
                   lambda m, i, t, k: compute_deep_eprop_gradients(
                       m, i, t, k, mse_error, d_zero=True)),
    'no_eps_z':   ('no \u03b5_z',  '#F28E2B',
                   lambda m, i, t, k: compute_no_eps_z_gradients(
                       m, i, t, k, mse_error)),
}


def train_rule_once(rule, seed, task_kw, n_rec, n_steps, eval_every,
                    batch_size, eval_batch_size, lr):
    model = make_model(2, n_rec, N_IN, N_OUT, seed)
    label, color, grad_fn = RULE_CFG[rule]
    steps, accs, losses = [], [], []

    def _eval(step):
        with torch.no_grad():
            ei, et, em = TXA.generate_batch(
                eval_batch_size, seed=8_000_000 + step, device=DEVICE, **task_kw)
            out, _ = model(ei)
            accs.append(TXA.task_accuracy(out, et, em))
            losses.append(float(_trace_mse_loss(out, et, em).item()))
            steps.append(step)

    _eval(0)
    for step in range(1, n_steps + 1):
        inp, tgt, msk = TXA.generate_batch(
            batch_size, seed=seed * 1_000_003 + step + 7777, device=DEVICE, **task_kw)
        grads = (compute_bptt_gradients(model, inp, tgt, msk, _trace_mse_loss)
                 if rule == 'bptt' else grad_fn(model, inp, tgt, msk))
        apply_grads(model, grads, lr)
        if step % eval_every == 0 or step == n_steps:
            _eval(step)

    return {'rule': rule, 'label': label, 'color': color, 'seed': seed,
            'steps': steps, 'accs': accs, 'losses': losses, 'final_acc': accs[-1]}


print('Running training curves ...')
t0 = time.time()
train_runs_all = []
for rule in RULE_CFG:
    for seed in range(CFG['n_seeds']):
        run = train_rule_once(
            rule=rule, seed=seed, task_kw=TASK_KW, n_rec=CFG['n_rec'],
            n_steps=CFG['n_steps'], eval_every=CFG['eval_every'],
            batch_size=CFG['batch_size'], eval_batch_size=CFG['eval_batch_size'],
            lr=CFG['lr'],
        )
        train_runs_all.append(run)
        print(f'  {rule:<12s}  seed={seed}  final acc={run["final_acc"]:.3f}')

print(f'Training done in {time.time() - t0:.1f}s')
with open(RESULTS_DIR / f'training_{RUN_PRESET}.json', 'w') as f:
    json.dump([{k: v for k, v in r.items()
                if not isinstance(v, torch.Tensor)} for r in train_runs_all], f, indent=2)


In [ ]:
def agg_runs(runs, rule):
    subset = [r for r in runs if r['rule'] == rule]
    mat_a  = np.array([r['accs']   for r in subset])
    mat_l  = np.array([r['losses'] for r in subset])
    xs     = subset[0]['steps']
    def _ms(m):
        mn = m.mean(0)
        se = m.std(0, ddof=1) / np.sqrt(len(m)) if len(m) > 1 else np.zeros_like(mn)
        return mn, se
    mn_a, se_a = _ms(mat_a)
    mn_l, se_l = _ms(mat_l)
    return {'steps': xs, 'acc_mean': mn_a, 'acc_se': se_a,
            'loss_mean': mn_l, 'loss_se': se_l}


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
LINE_STYLES = {'bptt': '-', 'deep-eprop': '--', 'd=0': ':', 'no_eps_z': '-.'}
for rule, (label, color, _) in RULE_CFG.items():
    agg = agg_runs(train_runs_all, rule)
    ls  = LINE_STYLES.get(rule, '-')
    for ax, (mn, se, ylab) in zip(axes, [
        (agg['acc_mean'],  agg['acc_se'],  'Decision accuracy'),
        (agg['loss_mean'], agg['loss_se'], 'MSE loss'),
    ]):
        ax.plot(agg['steps'], mn, color=color, label=label, linestyle=ls, linewidth=1.8)
        ax.fill_between(agg['steps'], mn - se, mn + se, color=color, alpha=0.15)

axes[0].axhline(0.5, color='0.5', linestyle=':', linewidth=1)
axes[0].set_ylim(0.3, 1.05)
for ax, ylab in zip(axes, ['Decision accuracy', 'MSE loss']):
    ax.set_xlabel('Training step'); ax.set_ylabel(ylab); ax.legend(fontsize=9)
    ax.set_title(f'Training curves \u2014 delay={CFG["fixed_delay"]}')
fig.tight_layout()
fig.savefig(RESULTS_DIR / f'training_{RUN_PRESET}.svg')
plt.show()


## PRIMARY: Layer-resolved gradient cosine vs BPTT, swept over delay D

For random-init depth-2 networks, compute the cosine similarity between each rule's gradient and the BPTT reference, broken down by layer group, across delays D.

**Key predictions**:
- **Bottom layer**: deep e-prop cosine > no_ε_z cosine (no_ε_z is exactly 0 by construction — all bottom-layer gradients come from ε_z).
- **Top layer**: deep e-prop ≈ no_ε_z (neither rule needs the cross-depth path for the top layer).
- Both bottom-layer cosines may decay with D as the eligibility trace accumulates approximation error over longer temporal horizons.


In [ ]:
def gradient_cosines_at_delay(delay, task_kw, n_trials, n_rec, batch_size):
    records = []
    kw = dict(task_kw, delay=delay)
    for trial in range(n_trials):
        trial_seed = 1_111 + delay * 10_000 + trial
        model = make_model(2, n_rec, N_IN, N_OUT, trial_seed).cpu()
        inp, tgt, msk = TXA.generate_batch(batch_size, seed=2_000_000 + trial_seed, **kw)

        g_bptt  = compute_bptt_gradients(model, inp, tgt, msk, _trace_mse_loss)
        g_eprop = compute_deep_eprop_gradients(model, inp, tgt, msk, mse_error, d_zero=False)
        g_d0    = compute_deep_eprop_gradients(model, inp, tgt, msk, mse_error, d_zero=True)
        g_nepsz = compute_no_eps_z_gradients(model, inp, tgt, msk, mse_error)

        for grp_name, keys in GROUPS_2L.items():
            if grp_name in ('hidden_all', 'readout', 'all'):
                continue
            records.append({
                'delay': delay, 'trial': trial, 'group': grp_name,
                'cos_eprop': cosine_for_keys(g_eprop, g_bptt, keys),
                'cos_d0':    cosine_for_keys(g_d0,    g_bptt, keys),
                'cos_nepsz': cosine_for_keys(g_nepsz, g_bptt, keys),
            })
    return records


print('Running cosine sweep ...')
t0 = time.time()
cosine_records = []
for delay in CFG['delays_cosine']:
    recs = gradient_cosines_at_delay(
        delay, TASK_KW, CFG['n_trials_cosine'], CFG['n_rec'], CFG['batch_size'])
    cosine_records.extend(recs)
    for grp in ('bottom', 'top'):
        sub = [r for r in recs if r['group'] == grp]
        ep  = np.nanmean([r['cos_eprop'] for r in sub])
        nep = np.nanmean([r['cos_nepsz'] for r in sub])
        d0  = np.nanmean([r['cos_d0']    for r in sub])
        print(f'  delay={delay:4d}  {grp:<7s}  ep={ep:.3f}  no_eps_z={nep:.3f}  d0={d0:.3f}')

print(f'Cosine sweep done in {time.time() - t0:.1f}s')
with open(RESULTS_DIR / f'cosine_{RUN_PRESET}.json', 'w') as f:
    json.dump(cosine_records, f, indent=2)


In [ ]:
def summarize_cosine(records, group):
    by_d: Dict = {}
    for r in records:
        if r['group'] != group:
            continue
        by_d.setdefault(r['delay'], {'ep': [], 'd0': [], 'nep': []})
        by_d[r['delay']]['ep'].append(r['cos_eprop'])
        by_d[r['delay']]['d0'].append(r['cos_d0'])
        by_d[r['delay']]['nep'].append(r['cos_nepsz'])
    delays = sorted(by_d)
    def _ms(vals):
        a = np.array([v for v in vals if not np.isnan(v)])
        if len(a) == 0:
            return float('nan'), 0.0
        se = a.std(ddof=1) / math.sqrt(len(a)) if len(a) > 1 else 0.0
        return float(a.mean()), float(se)
    return (delays,
            [_ms(by_d[d]['ep'])  for d in delays],
            [_ms(by_d[d]['d0'])  for d in delays],
            [_ms(by_d[d]['nep']) for d in delays])


fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, grp_label in zip(axes, [('bottom', 'Bottom layer'), ('top', 'Top layer')]):
    grp, title = grp_label
    delays, ep_v, d0_v, nep_v = summarize_cosine(cosine_records, grp)

    def _plot(vals, color, label, ls='-'):
        mn = np.array([v[0] for v in vals])
        se = np.array([v[1] for v in vals])
        ax.plot(delays, mn, color=color, label=label, linestyle=ls,
                marker='o', markersize=5, linewidth=1.8)
        ax.fill_between(delays, mn - se, mn + se, color=color, alpha=0.18)

    _plot(ep_v,  '#54A24B', 'Deep e-prop',  '-')
    _plot(nep_v, '#F28E2B', 'no \u03b5_z', '--')
    _plot(d0_v,  '#E45756', 'd=0',          ':')
    ax.axhline(0.0, color='0.4', linestyle=':', linewidth=0.9)
    ax.set_xlabel('Delay D'); ax.set_ylabel('Cosine vs BPTT')
    ax.set_title(title); ax.legend(fontsize=9)

fig.suptitle('Layer-resolved gradient cosine vs BPTT \u2014 temporal XOR task', fontsize=12)
fig.tight_layout()
fig.savefig(RESULTS_DIR / f'cosine_{RUN_PRESET}.svg')
plt.show()


## Summary


In [ ]:
d1_final = np.mean([r['final_acc'] for r in screen_result['d1_runs']])
d2_final = np.mean([r['final_acc'] for r in screen_result['d2_runs']])
max_delay = max(CFG['delays_cosine'])

ep_max  = [r for r in cosine_records if r['delay'] == max_delay and r['group'] == 'bottom']
ep_cos  = np.nanmean([r['cos_eprop'] for r in ep_max]) if ep_max else float('nan')
nep_cos = np.nanmean([r['cos_nepsz'] for r in ep_max]) if ep_max else float('nan')
d0_cos  = np.nanmean([r['cos_d0']    for r in ep_max]) if ep_max else float('nan')

final_accs = {rule: agg_runs(train_runs_all, rule) for rule in RULE_CFG}

lines = [
    (f"Task: temporal XOR, n_cues={TASK_KW['n_cues']}, w={TASK_KW['w']}, "
     f"gap={TASK_KW['gap']}, noise={TASK_KW['noise_level']}, "
     f"delay range 0..{max_delay}."),
    (f"Gate 1 (depth screen): depth=1={d1_final:.3f}, depth=2={d2_final:.3f}, "
     f"d2_frac_above_threshold={screen_result['d2_frac']:.0%} -- "
     + ('PASSED' if screen_result['passed'] else 'FAILED') + '.'),
    ('Training at fixed delay={}: BPTT={:.3f}, deep-eprop={:.3f}, '
     'd=0={:.3f}, no_eps_z={:.3f} (mean final acc).').format(
         CFG['fixed_delay'],
         final_accs['bptt']['acc_mean'][-1],
         final_accs['deep-eprop']['acc_mean'][-1],
         final_accs['d=0']['acc_mean'][-1],
         final_accs['no_eps_z']['acc_mean'][-1]),
    (f"Bottom-layer cosine at D={max_delay}: "
     f"deep-eprop={ep_cos:.3f}, no_eps_z={nep_cos:.3f}, d=0={d0_cos:.3f}."),
    ('eps_z separation confirmed (deep-eprop bottom >> no_eps_z).'
     if ep_cos > 0.05 else
     'eps_z separation NOT confirmed -- see cosine plot.'),
    f"Results: {sorted(RESULTS_DIR.glob(f'*_{RUN_PRESET}.*'))}",
]

print('\n' + '=' * 72)
print('EXPERIMENT SUMMARY')
print('=' * 72)
for ln in lines:
    print(ln)

with open(RESULTS_DIR / f'summary_{RUN_PRESET}.txt', 'w') as f:
    f.write('\n'.join(lines) + '\n')
print('\nSummary saved.')
